In [1]:
import sys, os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)
print('Project root:', project_root)

Project root: /Users/somita/Hospital_Supply_Chain_Bot


In [2]:
import pandas as pd
import sqlite3
from src.config import DATA_PATH, DB_PATH
print('Data:', DATA_PATH)
print('DB  :', DB_PATH)

Data: /Users/somita/Hospital_Supply_Chain_Bot/data
DB  : /Users/somita/Hospital_Supply_Chain_Bot/hospital.db


In [3]:
inv = pd.read_csv(DATA_PATH / 'inventory_data_clean.csv')
fin = pd.read_csv(DATA_PATH / 'financial_data_clean.csv')
ven = pd.read_csv(DATA_PATH / 'vendor_data_clean.csv')
pat = pd.read_csv(DATA_PATH / 'patient_data_clean.csv')
sta = pd.read_csv(DATA_PATH / 'staff_data_clean.csv')
for name, df in [('Inventory',inv),('Financial',fin),('Vendor',ven),('Patient',pat),('Staff',sta)]:
    print(f'{name:12s}: {df.shape[0]:4d} rows  {df.shape[1]:2d} cols')

Inventory   :  500 rows  16 cols
Financial   :  500 rows   6 cols
Vendor      :    9 rows   8 cols
Patient     :  500 rows  13 cols
Staff       :  500 rows  14 cols


In [4]:
# Computed columns
inv['Restock_Urgency_Days'] = inv['Days_Until_Stockout'] - inv['Restock_Lead_Time']
inv['Urgency_Level'] = inv['Restock_Urgency_Days'].apply(
    lambda x: 'CRITICAL' if x < 0 else ('WARNING' if x < 7 else 'OK'))
ven['Delay_Days'] = ven['Lead_Time_Actual_Days'] - ven['Avg_Lead_Time_days']
ven['Vendor_Status'] = ven['Delay_Days'].apply(
    lambda x: 'DELAYED' if x > 0 else 'ON_TIME')
print(inv['Urgency_Level'].value_counts())
print(ven[['Vendor_Name','Delay_Days','Vendor_Status']])

Urgency_Level
CRITICAL    298
OK          126
WARNING      76
Name: count, dtype: int64
        Vendor_Name  Delay_Days Vendor_Status
0  MedSupplies Inc.           0       ON_TIME
1      EquipMed Co.          14       DELAYED
2  HealthTools Ltd.           5       DELAYED
3  MedSupplies Inc.           0       ON_TIME
4  MedSupplies Inc.           0       ON_TIME
5  MedSupplies Inc.           0       ON_TIME
6      EquipMed Co.          14       DELAYED
7      EquipMed Co.          14       DELAYED
8  HealthTools Ltd.           5       DELAYED


In [6]:
conn = sqlite3.connect(str(DB_PATH))
inv.to_sql('inventory', conn, if_exists='replace', index=False)
fin.to_sql('financial', conn, if_exists='replace', index=False)
ven.to_sql('vendor',    conn, if_exists='replace', index=False)
pat.to_sql('patient',   conn, if_exists='replace', index=False)
sta.to_sql('staff',     conn, if_exists='replace', index=False)
conn.close()
print(f'Done. hospital.db → {DB_PATH}')

Done. hospital.db → /Users/somita/Hospital_Supply_Chain_Bot/hospital.db


In [7]:
from src.database import get_kpi_counts, get_critical_items
print(get_kpi_counts())
print(get_critical_items())

{'critical': 298, 'restock': 299, 'delayed': 5, 'total_spend_millions': 12.4}
       Item_Name  Current_Stock  Days_Until_Stockout  Restock_Lead_Time  \
0  Surgical Mask             69                  0.2                 14   
1  X-Ray Machine            139                  0.3                 19   
2         Gloves            156                  0.3                 24   
3  X-Ray Machine             93                  0.3                  2   
4  Surgical Mask            160                  0.3                 17   

   Restock_Urgency_Days Vendor_ID  
0                 -13.8      V001  
1                 -18.7      V003  
2                 -23.7      V001  
3                  -1.7      V003  
4                 -16.7      V001  
